# 08 · Model A: physics-structured outputs (the default model)

*Level: advanced. Guide Section 12. Code: `structured.PhysicsStructured`, `structured.StructuredPrep`, `structured.allan_fit`.*

**Idea.** Do not learn σe from scratch: the traces **measure** the noise in ADU, including how it averages down with skips and the part that does
not. Only the gain cannot be seen on a trace. So model A:
* takes the noise from the trace;
* learns the gain;
* learns a small correction for what the image sees and the trace does not.

A model that learns less needs fewer packets.

In [ ]:
from tutorial_utils import *

from ml import structured as st

## 1. Skipper averaging and the noise floor

Averaging $N$ independent skips of noise $\sigma_1$ gives $\sigma_1/\sqrt N$. Slow ($1/f$) noise is correlated from skip to skip and does **not**
average down, which leaves a floor:

$$\sigma_e^2 = \frac{\sigma_1^2}{N} + \sigma_\text{floor}^2 .$$

In [ ]:
rng = np.random.default_rng(0)
n_pix, n_skip = 400, 2000
white = rng.normal(0, 1.0, (n_pix, n_skip))
freqs = np.fft.rfftfreq(n_skip); shape = np.r_[0, 1 / np.sqrt(freqs[1:])]
flick = np.fft.irfft((rng.normal(size=(n_pix, freqs.size)) + 1j * rng.normal(size=(n_pix, freqs.size))) * shape, n=n_skip, axis=1)
flick *= 0.3 / flick.std()
skips = white + flick
Ns = np.array([1, 4, 16, 64, 256, 1000])
sd_pixel = [skips[:, :N].mean(1).std() for N in Ns]
plt.figure(figsize=(5.5, 3.2)); plt.loglog(Ns, sd_pixel, "ko", label="noise of the N-skip average")
plt.loglog(Ns, 1 / np.sqrt(Ns), "C0--", label="1/√N (white only)"); plt.xlabel("N skips"); plt.legend(); plt.show()

## 2. The Allan deviation, by hand

Split a pixel's skips into consecutive blocks of $n$, with block means $\bar y_k(n)$. Then

$$\sigma_A^2(n) = \tfrac12\,\mathbb E\bigl[(\bar y_{k+1}(n) - \bar y_k(n))^2\bigr].$$

* White noise gives $\sigma_A^2 = \sigma_1^2/n$.
* $1/f$ noise gives a flat $\sigma_A^2$.

Hence the fit $\sigma_A^2(n) = a^2/n + b^2$: it is linear in $1/n$, so it can be solved by (weighted) least squares.

In [ ]:
def allan(x, ns):
    out = []
    for n in ns:
        m = x.shape[1] // n
        blocks = x[:, :m * n].reshape(x.shape[0], m, n).mean(2)
        out.append(np.sqrt(0.5 * np.mean(np.diff(blocks, axis=1) ** 2)))
    return np.array(out)
ns = synthetic.ALLAN_N
sA = allan(skips, ns)
A_ = np.column_stack([1 / ns, np.ones_like(ns, dtype=float)]); w = 1 / sA ** 2          # weights for a constant relative scatter
coef = np.linalg.lstsq(A_ * w[:, None], sA ** 2 * w, rcond=None)[0]
a_hand, b_hand = np.sqrt(coef)
a_lib, b_lib = st.allan_fit(sA[None], ns)
print(f"by hand: a = {a_hand:.3f}, b = {b_hand:.3f}   |   structured.allan_fit: a = {a_lib[0]:.3f}, b = {b_lib[0]:.3f}")
g = np.geomspace(1, 200, 100)
plt.figure(figsize=(5.5, 3.2)); plt.loglog(ns, sA, "ko"); plt.loglog(g, np.sqrt(a_hand ** 2 / g + b_hand ** 2), "C3")
plt.loglog(g, a_hand / np.sqrt(g), "C0--"); plt.axhline(b_hand, color="C2", ls="--"); plt.xlabel("block length n"); plt.show()

## 3. Model A step by step on the toy CCD

1. $a, b$ from each trace's Allan curve, and from them the noise in ADU after $N$ skips: $\tfrac12\log(a^2/N + b^2)$.
2. Divide by the gain, $\log\sigma_e = \tfrac12\log(a^2/N + b^2) - \log g$.
3. With the **true** gain, how good is the trace alone? With the average gain (no knowledge of the amplifier)?

In [ ]:
df, curves, Y, groups = toy_scan(60, seed=1)
a, b = st.allan_fit(curves["noise: Allan deviation"], synthetic.ALLAN_N)
phys = 0.5 * np.log(a ** 2 / synthetic.N_SKIPS + b ** 2)
truth = np.log(df["truth.sigma_e"].to_numpy())
err = lambda p: 100 * np.sqrt(np.mean((p - truth) ** 2))
print(f"trace physics with the TRUE gain:    σe error {err(phys - np.log(df['truth.gain'])):.1f} %")
print(f"trace physics with the MEAN gain:    σe error {err(phys - Y[:, 2].mean()):.1f} %   (the gains differ between amplifiers)")

So model A learns:
* **the gain** $\hat g(x)$: a GP;
* **a correction** $\hat r(x)$: a GP trained on the residual $\log\sigma_e - \tfrac12\log(a^2/N+b^2) + \log g$ (with the *labelled* gain), i.e.
  whatever the image sees and the trace does not;
* λ: a GP on $\log\lambda$.

$$\log\sigma_e = \tfrac12\log\!\Bigl(\frac{a^2}{N} + b^2\Bigr) - \log\hat g(x) + \hat r(x).$$

**Inputs** $x$:
* the settings;
* the **packet state vector**: each reference feature minus that amplifier's average, averaged over the packet's amplifiers (the CCD's drifting state);
* the amplifier's **own** trace and reference features.

There is **no amplifier label**.

In [ ]:
dft, ct, Yt, gt = toy_scan(60, seed=2)                         # an independent test scan
truth_t = np.log(dft[["truth.sigma_e", "truth.lambda", "truth.gain"]].to_numpy())
for n in (15, 60):
    dtr, ctr, Ytr, _ = toy_scan(n, seed=1)
    prep = st.StructuredPrep().fit(dtr)
    A = st.PhysicsStructured().fit(prep.transform(dtr, ctr), Ytr)
    m, s = A.predict(prep.transform(dft, ct))
    _ = report(f"model A, {n} training packets (vs truth)", truth_t, m, s)
    Xtr, _ = data.make_inputs(dtr, ("theta", "trace", "ref"), drop_constant=False)
    Xte, _ = data.make_inputs(dft, ("theta", "trace", "ref"), drop_constant=False)
    m2, s2 = models.GP().fit(Xtr, Ytr).predict(Xte)
    _ = report(f"plain GP on features, {n} packets", truth_t, m2, s2)
print("\npacket state vector of the first test packet (first 5 numbers):", prep.state_vector(dft)[0, :5].round(3))

**Reading the numbers:**
* For σe, model A is far ahead with few packets (here ~3 % from 15 packets, against ~20 % for a GP that must learn everything).
* λ is hard for every model with 15 packets: it follows the CCD's drifting state, which takes more packets to learn. By 60 packets
  both models are close.
* The gain is learned well by both.

## When to use it

* **Good:** our first choice. It learns little, so it needs few packets. $N$ appears explicitly, so it predicts other skip numbers too.
* **Limits:**
  * it relies on the trace measuring the noise the image sees; level 1 tests exactly this on real data;
  * a never-seen amplifier needs one imaged packet for its gain, though the dependence on the settings is right without it.

Next: **09 · Hierarchical GP and amplifier labels.**